In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import HGate, CXGate
from qiskit_aer import AerSimulator
import numpy as np

# --------------------------------------------------
# Qubit Layout:
# Q0 - Q8: Data Qubits (9 total)
# A0 - A3: Ancilla Qubits (4 total for syndrome measurement)
# --------------------------------------------------
DATA_QUBITS = 9
ANCILLA_QUBITS = 4
TOTAL_QUBITS = DATA_QUBITS + ANCILLA_QUBITS

def shor_encode(qc: QuantumCircuit):
    """Encodes the logical state |psi> (on Q0) into the 9-qubit Shor code."""
    
    # 1. Phase Flip Encoding (Q0 -> Q0, Q3, Q6)
    qc.h([0, 3, 6])
    qc.cx(0, 3)
    qc.cx(0, 6)

    # 2. Bit Flip Encoding for Block 1 (Q0, Q1, Q2)
    qc.cx(0, 1)
    qc.cx(0, 2)

    # 3. Bit Flip Encoding for Block 2 (Q3, Q4, Q5)
    qc.cx(3, 4)
    qc.cx(3, 5)

    # 4. Bit Flip Encoding for Block 3 (Q6, Q7, Q8)
    qc.cx(6, 7)
    qc.cx(6, 8)

    return qc

# Initialize the circuit
#qc_full = QuantumCircuit(TOTAL_QUBITS, name="Shor_QEC")

CLASSICAL_BITS = 5 
qc_full = QuantumCircuit(TOTAL_QUBITS, CLASSICAL_BITS, name="Shor_QEC")

# Prepare a test state (e.g., |+> for a robust phase check)
qc_full.h(0) 

# Apply the encoding
qc_full = shor_encode(qc_full)

In [6]:
# --------------------------------------------------
# ERROR STAGE: Introduce a Z-error (Phase Error) on Qubit 1
# --------------------------------------------------

# The error could be X, Y, or Z. We'll use Z for a clear phase error test.
ERROR_QUBIT = 1 
print(f"--- Introducing Z-error on Qubit {ERROR_QUBIT} ---")

qc_full.barrier()
qc_full.z(ERROR_QUBIT) # Z-error (Phase-flip error)
qc_full.barrier()

--- Introducing Z-error on Qubit 1 ---


CircuitInstruction(operation=Instruction(name='barrier', num_qubits=13, num_clbits=0, params=[]), qubits=(<Qubit register=(13, "q"), index=0>, <Qubit register=(13, "q"), index=1>, <Qubit register=(13, "q"), index=2>, <Qubit register=(13, "q"), index=3>, <Qubit register=(13, "q"), index=4>, <Qubit register=(13, "q"), index=5>, <Qubit register=(13, "q"), index=6>, <Qubit register=(13, "q"), index=7>, <Qubit register=(13, "q"), index=8>, <Qubit register=(13, "q"), index=9>, <Qubit register=(13, "q"), index=10>, <Qubit register=(13, "q"), index=11>, <Qubit register=(13, "q"), index=12>), clbits=())

In [7]:
# Ancilla Qubit Indices
A0, A1, A2, A3 = 9, 10, 11, 12

# --------------------------------------------------
# PHASE FLIP SYNDROME MEASUREMENT (S4: Identifies the block with Z/Y error)
# --------------------------------------------------

# Measures Z-syndrome (which of the 3 blocks has the phase error)
# Block 1 vs Block 2 (Ancilla A2)
qc_full.h([A2, 0, 3])
qc_full.cz(0, A2) 
qc_full.cz(3, A2) 
qc_full.h([A2, 0, 3])
qc_full.cx(0, A2) # Using CX here simplifies the final correction logic below, though CZ is standard

# Block 2 vs Block 3 (Ancilla A3)
qc_full.h([A3, 3, 6])
qc_full.cz(3, A3)
qc_full.cz(6, A3)
qc_full.h([A3, 3, 6])
qc_full.cx(3, A3)

# Correction for Phase Error: The measured syndrome identifies the block.
# We apply X and CX to transfer the Z-error location to a single qubit (Q0).
# In this specific simplified implementation, we rely on the final measurement.

# --------------------------------------------------
# BIT FLIP SYNDROME MEASUREMENT (S1, S2, S3: Identifies the qubit in the block)
# --------------------------------------------------

# Bit flip syndrome is done on the three blocks using A0 and A1 (Q9, Q10)
# Measure Q0, Q1, Q3, Q4, Q6, Q7 using A0 (Q9) and A1 (Q10)

qc_full.cx(0, A0)
qc_full.cx(1, A0)
qc_full.cx(0, A1)
qc_full.cx(2, A1)

# *Note: A full implementation would measure the other two blocks, 
# correct the error, and then perform the phase correction. 
# For demonstration, we focus on measuring the syndromes to get the error bits.*

In [8]:
# --------------------------------------------------
# DECODING AND VERIFICATION
# --------------------------------------------------

# --- Reverse Bit Flip Encoding ---
qc_full.cx(0, 2)
qc_full.cx(0, 1)

qc_full.cx(3, 5)
qc_full.cx(3, 4)

qc_full.cx(6, 8)
qc_full.cx(6, 7)

# --- Reverse Phase Flip Encoding ---
qc_full.cx(0, 6)
qc_full.cx(0, 3)
qc_full.h([0, 3, 6])


# --------------------------------------------------
# MEASUREMENT
# --------------------------------------------------

# Measure the Ancilla qubits to read the syndrome (Error location)
# Measure the Data qubit Q0 to check the logical state recovery
qc_full.measure([A0, A1, A2, A3, 0], [0, 1, 2, 3, 4]) 

# Set up the simulation
simulator = AerSimulator()
qc_transpiled = transpile(qc_full, simulator)

# Run the simulation (1024 times)
job = simulator.run(qc_transpiled, shots=1024)
counts = job.result().get_counts()

# --------------------------------------------------
# RESULTS ANALYSIS
# --------------------------------------------------

print("\n" + "="*50)
print("### 9-Qubit Shor Code Simulation Results ###")
print(f"Original Logical State: |+⟩ (Expected Q0=0 or 1, 50/50)")
print(f"Induced Error: Z-error on Qubit {ERROR_QUBIT}")
print(f"Total Qubits: {TOTAL_QUBITS}")
print("="*50)

# The output format is (c4 c3 c2 c1 c0), where c4 is Q0 measurement, c0 is A0 measurement.
# We focus on the measurement of the logical qubit Q0 (classical bit c4).
recovered_0_count = sum(count for outcome, count in counts.items() if outcome[0] == '0')
recovered_1_count = sum(count for outcome, count in counts.items() if outcome[0] == '1')
total_shots = sum(counts.values())

print(f"Counts (Total Shots: {total_shots}): {counts}")
print("\n--- Logical Qubit Recovery (Q0, classical bit 4) ---")
print(f"P(|0⟩): {recovered_0_count / total_shots:.4f}")
print(f"P(|1⟩): {recovered_1_count / total_shots:.4f}")

# Ideal outcome: If the correction worked, the state |+> should be recovered, 
# meaning P(|0>) should be ~0.5 and P(|1>) should be ~0.5.
if np.isclose(recovered_0_count / total_shots, 0.5, atol=0.05):
    print("✅ **SUCCESS!** Logical state recovered (P(|0>) ≈ 50%).")
else:
    print("❌ **FAILURE!** Logical state NOT recovered.")

print("\n--- Ancilla Syndrome (Q9-Q12, classical bits 0-3) ---")
print("These measurements identify the error location.")

# For a Z-error on Q1, the bit-flip syndrome should be 00 and the phase-flip 
# syndrome should indicate the first block.


### 9-Qubit Shor Code Simulation Results ###
Original Logical State: |+⟩ (Expected Q0=0 or 1, 50/50)
Induced Error: Z-error on Qubit 1
Total Qubits: 13
Counts (Total Shots: 1024): {'01111': 54, '10011': 53, '01100': 69, '00100': 68, '01011': 67, '10100': 78, '00111': 65, '00011': 64, '10000': 70, '11011': 71, '00000': 53, '01000': 54, '10111': 54, '11100': 63, '11111': 73, '11000': 68}

--- Logical Qubit Recovery (Q0, classical bit 4) ---
P(|0⟩): 0.4824
P(|1⟩): 0.5176
✅ **SUCCESS!** Logical state recovered (P(|0>) ≈ 50%).

--- Ancilla Syndrome (Q9-Q12, classical bits 0-3) ---
These measurements identify the error location.
